In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import random
from PIL import Image
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
# Define the path to your zip file in Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/Solidworks_Hackathon/dataset/dataset.zip'
# Define where to extract in Colab (Local RAM)
LOCAL_DIR = '/content/dataset'

# --- 1. UNZIP DATA ---
if not os.path.exists(LOCAL_DIR):
    print(" Unzipping dataset... (This might take 1-2 mins)")
    # -q means quiet (no huge log), -d is destination
    !unzip -q "$DRIVE_ZIP_PATH" -d "$LOCAL_DIR"
    print(" Unzipping Complete!")
else:
    print(" Dataset already exists locally.")

# --- 2. INSPECT STRUCTURE ---
print("\n File Structure inside /content/dataset:")
for root, dirs, files in os.walk(LOCAL_DIR):
    level = root.replace(LOCAL_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    # Print first 3 files as examples
    for f in files[:3]:
        print(f"{subindent}{f}")
    if len(files) > 3:
        print(f"{subindent}... ({len(files)-3} more files)")

Mounted at /content/drive
 Unzipping dataset... (This might take 1-2 mins)
 Unzipping Complete!

 File Structure inside /content/dataset:
dataset/
    train_bboxes.csv
    train_labels.csv
    sample_submission.csv
    train/
        train/
            4c4fe793c8744f1aa9f67928abaa6142.png
            0bbb56af263e4d2f9561df9d788364fb.png
            55f5a45d4bc14caf8e31b3fccf515645.png
            ... (9997 more files)
    test/
        test/
            80c1c539c8af458498728c85a5e36522.png
            9734b5f01e27480682d6b73dcf6f73b6.png
            65116b3f90b44602a870f309926ce1ad.png
            ... (1997 more files)


In [4]:
import pandas as pd
import os
import shutil
import glob
from tqdm.notebook import tqdm
from PIL import Image

# --- CONFIGURATION ---
BASE_DIR = '/content/dataset'
BBOX_CSV = os.path.join(BASE_DIR, 'train_bboxes.csv')
TRAIN_IMGS_DIR = os.path.join(BASE_DIR, 'train/train') # Update if your path is different
YOLO_DIR = '/content/yolo_dataset'

class_map = {'bolt': 0, 'locatingpin': 1, 'nut': 2, 'washer': 3}

# 1. SETUP FOLDERS
for split in ['train', 'val']:
    os.makedirs(f'{YOLO_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_DIR}/labels/{split}', exist_ok=True)

# 2. READ DATA
print(f" Reading {BBOX_CSV}...")
df = pd.read_csv(BBOX_CSV)

# Get ALL training images (even those without boxes)
all_img_files = glob.glob(os.path.join(TRAIN_IMGS_DIR, '*.png'))
all_img_names = [os.path.basename(f) for f in all_img_files]
print(f" Found {len(all_img_names)} total images in folder.")

# Split: 90% Train, 10% Validation
split_idx = int(len(all_img_names) * 0.9)
train_imgs = all_img_names[:split_idx]
val_imgs = all_img_names[split_idx:]

# 3. CONVERSION FUNCTION
def prepare_yolo_data(image_list, split_name):
    print(f" Processing {split_name} data...")

    for img_name in tqdm(image_list):
        src_path = os.path.join(TRAIN_IMGS_DIR, img_name)
        if not os.path.exists(src_path): continue

        # Move Image
        dst_img_path = f'{YOLO_DIR}/images/{split_name}/{img_name}'
        shutil.copy(src_path, dst_img_path)

        # Prepare Label Path
        label_filename = os.path.splitext(img_name)[0] + '.txt'
        label_path = f'{YOLO_DIR}/labels/{split_name}/{label_filename}'

        # Check if this image has boxes
        img_boxes = df[df['image_name'] == img_name]

        # OPEN FILE (Creates empty file if no boxes, which is GOOD)
        with open(label_path, 'w') as f:
            if len(img_boxes) > 0:
                with Image.open(src_path) as im:
                    img_w, img_h = im.size

                for _, row in img_boxes.iterrows():
                    cls_name = row['class']
                    if cls_name not in class_map: continue
                    cls_id = class_map[cls_name]

                    # Normalize Coordinates
                    xmin, ymin = row['x_min'], row['y_min']
                    xmax, ymax = row['x_max'], row['y_max']

                    w = xmax - xmin
                    h = ymax - ymin
                    x_center = xmin + (w / 2.0)
                    y_center = ymin + (h / 2.0)

                    x_c_norm = x_center / img_w
                    y_c_norm = y_center / img_h
                    w_norm = w / img_w
                    h_norm = h / img_h

                    f.write(f"{cls_id} {x_c_norm:.6f} {y_c_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# 4. RUN
prepare_yolo_data(train_imgs, 'train')
prepare_yolo_data(val_imgs, 'val')
print(" Data Prep Complete (Included empty images for robustness).")

 Reading /content/dataset/train_bboxes.csv...
 Found 10000 total images in folder.
 Processing train data...


  0%|          | 0/9000 [00:00<?, ?it/s]

 Processing val data...


  0%|          | 0/1000 [00:00<?, ?it/s]

 Data Prep Complete (Included empty images for robustness).


In [5]:
import yaml

data_config = {
    'path': '/content/yolo_dataset',  # Root dir
    'train': 'images/train',          # Train images
    'val': 'images/val',              # Val images
    'names': {                        # Class names
        0: 'bolt',
        1: 'locatingpin',
        2: 'nut',
        3: 'washer'
    }
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print(" data.yaml created.")

 data.yaml created.


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.6 MB/s eta 0:00:00


In [ ]:
import os
import glob
from ultralytics import YOLO

# --- CONFIGURATION ---
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Solidworks_Hackathon/models'
BASE_RUN_NAME = 'yolo_run' # Folders will be run1, run2, run3...
DATA_YAML = '/content/data.yaml'
EPOCHS_PER_RUN =  60 # Each new run adds 60 epochs

# ---------------------------------------------------------
# 1. AUTOMATICALLY FIND THE NEXT RUN NUMBER
# ---------------------------------------------------------
# Find all folders like 'yolo_scratch_run1', 'yolo_scratch_run2'...
existing_folders = glob.glob(os.path.join(DRIVE_PROJECT_PATH, f"{BASE_RUN_NAME}*"))

# Extract the numbers from folder names
run_numbers = []
for folder in existing_folders:
    try:
        # Get the number at the end of the folder name
        folder_name = os.path.basename(folder)
        num = int(folder_name.replace(BASE_RUN_NAME, ''))
        run_numbers.append(num)
    except:
        pass # Ignore folders that don't match the pattern

# Determine what to do next
if not run_numbers:
    # CASE A: No runs exist yet. Start Run 1.
    next_run_num = 1
    prev_weights = None
    print(" No previous runs found. Starting Run 1 FROM SCRATCH (.yaml)...")

else:
    # CASE B: Runs exist. Find the latest one.
    last_run_num = max(run_numbers)
    last_run_folder = os.path.join(DRIVE_PROJECT_PATH, f"{BASE_RUN_NAME}{last_run_num}")
    prev_weights = os.path.join(last_run_folder, 'weights', 'last.pt')

    # Check if weights exist
    if os.path.exists(prev_weights):
        next_run_num = last_run_num + 1
        print(f" Found Run {last_run_num}. Chaining to Run {next_run_num}...")
        print(f"   (Loading weights from: {prev_weights})")
    else:
        # Fallback: If Run 1 was created but has no weights (crashed instantly?), retry Run 1.
        print(f" Run {last_run_num} exists but has no weights! Retrying Run {last_run_num}...")
        next_run_num = last_run_num
        prev_weights = None # Treat as fresh start if weights are missing

# Define the new run name (e.g., yolo_scratch_run2)
NEW_RUN_NAME = f"{BASE_RUN_NAME}{next_run_num}"

# ---------------------------------------------------------
# 2. START TRAINING
# ---------------------------------------------------------

# Load the correct model
if prev_weights:
    model = YOLO(prev_weights) # Continue from last run
else:
    model = YOLO('yolov8s.yaml') # Start Fresh

# Train
print(f" Starting Training Session: {NEW_RUN_NAME}")
model.train(
    data=DATA_YAML,
    epochs=EPOCHS_PER_RUN,  # This runs 60 *NEW* epochs
    patience=0,             # Disable early stop for relay
    imgsz=640,
    batch=16,
    project=DRIVE_PROJECT_PATH,
    name=NEW_RUN_NAME,
    save_period=1,
    exist_ok=True
)

print(f" Session Ended. Results saved to: {NEW_RUN_NAME}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 No previous runs found. Starting Run 1 FROM SCRATCH (.yaml)...
 Starting Training Session: yolo_run1
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=N

In [ ]:
import os
import glob
from ultralytics import YOLO

# --- CONFIGURATION ---
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Solidworks_Hackathon/models'
BASE_RUN_NAME = 'yolo_run' # Folders will be run1, run2, run3...
DATA_YAML = '/content/data.yaml'
EPOCHS_PER_RUN =  60 # Each new run adds 60 epochs

# ---------------------------------------------------------
# 1. AUTOMATICALLY FIND THE NEXT RUN NUMBER
# ---------------------------------------------------------
# Find all folders like 'yolo_scratch_run1', 'yolo_scratch_run2'...
existing_folders = glob.glob(os.path.join(DRIVE_PROJECT_PATH, f"{BASE_RUN_NAME}*"))

# Extract the numbers from folder names
run_numbers = []
for folder in existing_folders:
    try:
        # Get the number at the end of the folder name
        folder_name = os.path.basename(folder)
        num = int(folder_name.replace(BASE_RUN_NAME, ''))
        run_numbers.append(num)
    except:
        pass # Ignore folders that don't match the pattern

# Determine what to do next
if not run_numbers:
    # CASE A: No runs exist yet. Start Run 1.
    next_run_num = 1
    prev_weights = None
    print(" No previous runs found. Starting Run 1 FROM SCRATCH (.yaml)...")

else:
    # CASE B: Runs exist. Find the latest one.
    last_run_num = max(run_numbers)
    last_run_folder = os.path.join(DRIVE_PROJECT_PATH, f"{BASE_RUN_NAME}{last_run_num}")
    prev_weights = os.path.join(last_run_folder, 'weights', 'last.pt')

    # Check if weights exist
    if os.path.exists(prev_weights):
        next_run_num = last_run_num + 1
        print(f" Found Run {last_run_num}. Chaining to Run {next_run_num}...")
        print(f"   (Loading weights from: {prev_weights})")
    else:
        # Fallback: If Run 1 was created but has no weights (crashed instantly?), retry Run 1.
        print(f" Run {last_run_num} exists but has no weights! Retrying Run {last_run_num}...")
        next_run_num = last_run_num
        prev_weights = None # Treat as fresh start if weights are missing

# Define the new run name (e.g., yolo_scratch_run2)
NEW_RUN_NAME = f"{BASE_RUN_NAME}{next_run_num}"

# ---------------------------------------------------------
# 2. START TRAINING
# ---------------------------------------------------------

# Load the correct model
if prev_weights:
    model = YOLO(prev_weights) # Continue from last run
else:
    model = YOLO('yolov8s.yaml') # Start Fresh

# Train
print(f" Starting Training Session: {NEW_RUN_NAME}")
model.train(
    data=DATA_YAML,
    epochs=EPOCHS_PER_RUN,  # This runs 60 *NEW* epochs
    patience=0,             # Disable early stop for relay
    imgsz=640,
    batch=16,
    project=DRIVE_PROJECT_PATH,
    name=NEW_RUN_NAME,
    save_period=1,
    exist_ok=True
)

print(f" Session Ended. Results saved to: {NEW_RUN_NAME}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Found Run 1. Chaining to Run 2...
   (Loading weights from: /content/drive/MyDrive/Solidworks_Hackathon/models/yolo_run1/weights/last.pt)
 Starting Training Session: yolo_run2
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_

In [7]:
from ultralytics import YOLO

# Load your best weights
model = YOLO('/content/drive/MyDrive/Solidworks_Hackathon/models/yolo_run2/weights/best.pt')

print("🚀 Running Validation with TTA (The 'Cheeky' Boost)...")

# augment=True turns on the TTA magic
metrics = model.val(data='/content/data.yaml', augment=True)

print(f"PRECISION: {metrics.box.mp}")
print(f"RECALL:    {metrics.box.mr}")
print(f"mAP50:     {metrics.box.map50}")

🚀 Running Validation with TTA (The 'Cheeky' Boost)...
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1179.4±595.6 MB/s, size: 23.4 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 2.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.3it/s 27.9s
                   all       1000       2594          1          1      0.995      0.994
                  bolt        483        621          1          1      0.995      0.995
           locatingpin        488        620          1          1      0.995      0.995
                   nut        482        627          1          1      0.995      0.995
                washer        533        726          1          1      0.99

In [8]:
!pip install ultralytics

import os
import glob
import pandas as pd
from ultralytics import YOLO
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
MODEL_PATH = '/content/drive/MyDrive/Solidworks_Hackathon/models/yolo_run2/weights/best.pt'
TEST_IMAGES_DIR = '/content/dataset/test/test'

# CRITICAL UPDATE: Point to the file you uploaded to DRIVE
SAMPLE_SUB_PATH = '/content/drive/MyDrive/Solidworks_Hackathon/dataset/sample_submission.csv'
SUBMISSION_PATH = '/content/drive/MyDrive/Solidworks_Hackathon/submission/final_submission3.csv'

# 1. LOAD MODEL & TEMPLATE
print(f" Loading model from {MODEL_PATH}...")
model = YOLO(MODEL_PATH)

print(f" Loading submission template from {SAMPLE_SUB_PATH}...")
if not os.path.exists(SAMPLE_SUB_PATH):
    raise FileNotFoundError(f" Could not find the CSV at: {SAMPLE_SUB_PATH}")

df_sub = pd.read_csv(SAMPLE_SUB_PATH)
print(f" Loaded template with {len(df_sub)} rows.")

# 2. INFERENCE LOOP (WITH TTA)
print(" Starting Inference with TTA (This will be slower but more accurate)...")

for index, row in tqdm(df_sub.iterrows(), total=len(df_sub)):
    filename = row['image_name']
    img_path = os.path.join(TEST_IMAGES_DIR, filename)

    # Safety: Handle missing images
    if not os.path.exists(img_path):
        print(f" WARNING: Image {filename} missing. Skipping.")
        continue

    # --- THE MAGIC LINE ---
    # augment=True enables Test Time Augmentation (TTA)
    results = model.predict(
        img_path,
        verbose=False,
        conf=0.25,      # Slight bump to 0.25 is usually safer for TTA
        augment=True    # <--- THIS IS THE KEY TO 1.0 SCORE
    )

    # Count Objects
    counts = {0: 0, 1: 0, 2: 0, 3: 0} # 0=bolt, 1=pin, 2=nut, 3=washer
    for r in results:
        for cls_id in r.boxes.cls:
            counts[int(cls_id)] += 1

    # Update DataFrame
    df_sub.at[index, 'bolt'] = counts[0]
    df_sub.at[index, 'locatingpin'] = counts[1]
    df_sub.at[index, 'nut'] = counts[2]
    df_sub.at[index, 'washer'] = counts[3]

# 3. SAVE
df_sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\n Submission saved to: {SUBMISSION_PATH}")

# ==========================================
#  SANITY CHECK BLOCK
# ==========================================
print("\n --- RUNNING SANITY CHECKS ---")
expected_cols = ['image_name', 'bolt', 'locatingpin', 'nut', 'washer']

if list(df_sub.columns) == expected_cols:
    print(" [PASS] Column names match perfectly.")
else:
    print(f" [FAIL] Column Mismatch! Expected {expected_cols}, Got {list(df_sub.columns)}")

if df_sub.isnull().values.any():
    print(" [FAIL] Found NaN values!")
else:
    print(" [PASS] No null values found.")

print("\n READY FOR SUBMISSION!")

 Loading model from /content/drive/MyDrive/Solidworks_Hackathon/models/yolo_run2/weights/best.pt...
 Loading submission template from /content/drive/MyDrive/Solidworks_Hackathon/dataset/sample_submission.csv...
 Loaded template with 2000 rows.
 Starting Inference with TTA (This will be slower but more accurate)...


  0%|          | 0/2000 [00:00<?, ?it/s]


 Submission saved to: /content/drive/MyDrive/Solidworks_Hackathon/submission/final_submission3.csv

 --- RUNNING SANITY CHECKS ---
 [PASS] Column names match perfectly.
 [PASS] No null values found.

 READY FOR SUBMISSION!
